[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/52_focal_loss_solution.ipynb)

# 🟡 Solution: Implement Focal Loss

Reference solution for focal loss.

$$FL(p_t) = -\alpha (1-p_t)^\gamma \log(p_t)$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    N, C = logits.shape
    shifted = logits - logits.max(dim=-1, keepdim=True).values
    exp_s = torch.exp(shifted)
    probs = exp_s / exp_s.sum(dim=-1, keepdim=True)
    p_t = probs[torch.arange(N), targets]
    log_p_t = torch.log(p_t + 1e-8)
    fl = -alpha * (1 - p_t) ** gamma * log_p_t
    return fl.mean()

In [ ]:
import torch.nn.functional as F

torch.manual_seed(0)
logits = torch.randn(8, 4)
targets = torch.randint(0, 4, (8,))

# gamma=0 should match weighted CE (alpha * CE)
fl_gamma0 = focal_loss(logits, targets, alpha=0.25, gamma=0.0)
ce = F.cross_entropy(logits, targets)
print(f"Focal (gamma=0): {fl_gamma0:.4f}  |  alpha * CE: {0.25 * ce:.4f}")

# Higher gamma should reduce loss on easy examples
fl_g2 = focal_loss(logits, targets, alpha=0.25, gamma=2.0)
fl_g5 = focal_loss(logits, targets, alpha=0.25, gamma=5.0)
print(f"Focal gamma=2: {fl_g2:.4f}  |  gamma=5: {fl_g5:.4f}  (higher gamma -> lower loss)")

In [ ]:
from torch_judge import check
check("focal_loss")